In [1]:
import json

with open("../data/EXIST 2026 Videos Dataset/training/metadata.json") as f:
    data = json.load(f)
with open("../data/EXIST 2026 Videos Dataset/training/qwen_verdicts_alc3_description.json") as f:
    qwen_data = json.load(f)
    

In [2]:
def get_verdict_from_cache(tiktok_id: str) -> str:
    return qwen_data.get(str(tiktok_id), "")

for meme_id, sample in data.items():
    tiktok_id = sample.get("id_Tiktok")
    sample["qwen_verdict"] = get_verdict_from_cache(tiktok_id)

print(f"✅ Verdicts cargados desde caché para {len(data)} memes.")

✅ Verdicts cargados desde caché para 2524 memes.


In [3]:
import numpy as np

# ── CORREGIDO ────────────────────────────────────────────────────────────────
# ANTES: se aplicaba RobustScaler + Z-score POR CADA SUJETO individualmente.
# PROBLEMA: normalizar por sujeto garantiza que todos los usuarios acaban con
#   media=0 y std=1, eliminando exactamente las diferencias de magnitud entre
#   memes sexistas y no sexistas (más fixations, más RT, diferente EEG power)
#   que el paper detectó como significativas (p<0.001).
# FIX: aquí solo limpiamos NaN/Inf. La normalización global (ajustada sobre
#   TODO el train set) se aplica en la Cell 7b, después del split.
# ─────────────────────────────────────────────────────────────────────────────

def extract_physio_features_raw(sensorial):
    """
    Extrae features fisiológicas sin normalizar.
    Solo limpia NaN/Inf para evitar errores numéricos en pasos posteriores.
    Devuelve: {"ET": [[vec_suj1], ...], "HR": [...], "EEG": [...]}
    """
    output = {}

    for modality in ["ET", "HR", "EEG"]:
        if modality not in sensorial.get("modalities", {}):
            continue

        users_data = sensorial["modalities"][modality]["by_user"]
        subject_vectors = []

        for user_id, features_dict in users_data.items():
            vec = np.array(
                [v if v is not None else np.nan for v in features_dict.values()],
                dtype=float
            )
            # Limpieza básica de NaN/Inf — sin normalizar
            if np.all(np.isnan(vec)):
                vec = np.zeros_like(vec)
            else:
                mean_val = np.nanmean(vec)
                vec = np.nan_to_num(vec, nan=mean_val, posinf=mean_val, neginf=mean_val)

            subject_vectors.append(vec.tolist())

        output[modality] = subject_vectors

    return output


In [4]:
import re

def parse_qwen_verdict(output: str) -> dict:
    transcription_match = re.search(r'TRANSCRIPTION:\s*(.*?)\nOCR:', output, re.DOTALL)
    ocr_match = re.search(r'OCR:\s*(.*?)\DESCRIPTION:', output, re.DOTALL)
    reasoning_match = re.search(r'DESCRIPTION:\s*(.*?)\nLABEL:', output, re.DOTALL)
    label_match = re.search(r'LABEL:\s*(YES.*?|NO.*?)\n?', output)

    return {
        "transcription": transcription_match.group(1).strip() if transcription_match else "",
        "ocr": ocr_match.group(1).strip() if ocr_match else "",
        "reasoning": reasoning_match.group(1).strip() if reasoning_match else "",
        "label": label_match.group(1).strip() if label_match else "",
    }

In [5]:
def prepare_labels(ls):
    aux = []
    for l in ls:
        if l == "YES":
            aux.append(1)
        else:
            aux.append(0)

    return aux

In [6]:
def clean_tiktok(texto):
    return re.sub(r"(TikTok\s*)?@\w+\s*", "", texto)


In [7]:
dataset = []

for meme_id, sample in data.items():
    id = int(sample["id_EXIST"])
    task1      = sample["labels_task3_1"]
    task1 = prepare_labels(task1)
    physio_raw = extract_physio_features_raw(sample["sensorial"])
    lang = sample.get("lang")

    qwen_output  = sample.get("qwen_verdict", "")
    parsed_qwen = parse_qwen_verdict(qwen_output)
 
    dataset.append({
        "id": id,
        "id_Tiktok": sample.get("id_Tiktok"),
        "physio": physio_raw,
        "label": task1,
        "lang": lang,
        "qwen_transcription": sample["transcription"],
        "qwen_ocr": clean_tiktok(sample["ocr"]),
        "qwen_reasoning":     parsed_qwen["reasoning"],
    })

In [8]:
import json
import random
random.seed(42)
import os
from scipy import stats
from sklearn.preprocessing import RobustScaler

# ── NUEVO: Normalización global por modalidad ─────────────────────────────────
# Por qué es necesario:
#   El paper detecta diferencias de magnitud entre memes sexistas y no sexistas:
#   más fixation count, mayor reaction time, diferente EEG power (p<0.001).
#   Si normalizamos por sujeto individualmente (como hacía el código original),
#   forzamos media=0 std=1 para cada usuario y destruimos esa señal.
#   La solución correcta es ajustar UN scaler global sobre todos los sujetos
#   del train set y aplicarlo igual a train y val (sin leakage).
# ─────────────────────────────────────────────────────────────────────────────

class GlobalPhysioNormalizer:
    """
    Normalización adaptada a las escalas reales de cada modalidad:
    - EEG: solo RobustScaler global (ya viene en escala razonable ~[-3, 3])
    - ET:  log1p en features de duración/tiempo + RobustScaler global
    - HR:  RobustScaler global (escala ya pequeña)
    """
    def __init__(self):
        self.scalers      = {}
        self.log_cols     = {}  # {modality: bool array} columnas que necesitan log
        self.clip_vals    = {}  # {modality: (lo, hi)} para outliers extremos
        self.fitted       = False

    # features de ET que están en nanosegundos o son conteos grandes
    ET_LOG_KEYWORDS = [
        "duration", "reaction_time", "count"
    ]

    def _needs_log(self, feature_names, modality):
        """Devuelve máscara booleana de qué columnas aplicar log1p."""
        if modality != "ET":
            return np.zeros(len(feature_names), dtype=bool)
        mask = np.array([
            any(kw in name.lower() for kw in self.ET_LOG_KEYWORDS)
            for name in feature_names
        ])
        return mask

    def _collect_all_vectors(self, dataset_list, modality):
        all_vecs = []
        for s in dataset_list:
            for vec in s["physio"].get(modality, []):
                if vec and not all(v == 0 for v in vec):
                    all_vecs.append(vec)
        return np.array(all_vecs, dtype=float) if all_vecs else None

    def _get_feature_names(self, dataset_list, modality):
        """Saca los nombres de features del primer sujeto que encuentre."""
        for s in dataset_list:
            raw = s.get("_physio_raw", {}).get(modality)
            if raw:
                return list(raw.keys())
        return []

    def fit(self, train_list):
        import warnings
        for modality in ["ET", "HR", "EEG"]:
            X = self._collect_all_vectors(train_list, modality)
            if X is None or X.shape[0] == 0:
                continue

            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

            # 1. Clip outliers extremos (solo para ET que tiene ns)
            pct = 2 if modality == "ET" else 1
            lo = np.percentile(X, pct, axis=0)
            hi = np.percentile(X, 100 - pct, axis=0)
            self.clip_vals[modality] = (lo, hi)
            X = np.clip(X, lo, hi)

            # 2. log1p solo en columnas de duración/conteo de ET
            #    (reaction_time en ms, durations en ns, counts)
            #    EEG y HR no necesitan transformación — ya están en escala razonable
            n_features = X.shape[1]
            if modality == "ET":
                # heurística: columnas con valores > 1000 probablemente son ns o ms
                col_max = np.max(np.abs(X), axis=0)
                log_mask = col_max > 1000
            else:
                log_mask = np.zeros(n_features, dtype=bool)

            self.log_cols[modality] = log_mask

            X_t = X.copy()
            if log_mask.any():
                # Asegurar positivo antes del log
                X_t[:, log_mask] = np.log1p(np.abs(X[:, log_mask]))

            X_t = np.nan_to_num(X_t, nan=0.0, posinf=0.0, neginf=0.0)

            # 3. RobustScaler global
            scaler = RobustScaler()
            scaler.fit(X_t)
            self.scalers[modality] = scaler

        self.fitted = True
        for mod in self.log_cols:
            n_log = self.log_cols[mod].sum()
            print(f"[GlobalPhysioNormalizer] {mod}: {n_log} columnas con log1p, "
                  f"{len(self.log_cols[mod]) - n_log} directas")

    def _transform_vec(self, vec, modality):
        lo, hi   = self.clip_vals[modality]
        log_mask = self.log_cols[modality]
        scaler   = self.scalers[modality]

        vec = np.array(vec, dtype=float)
        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
        vec = np.clip(vec, lo, hi)

        if log_mask.any():
            vec[log_mask] = np.log1p(np.abs(vec[log_mask]))

        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
        return scaler.transform(vec.reshape(1, -1)).flatten().tolist()

    def transform_dataset(self, dataset_list):
        assert self.fitted
        for sample in dataset_list:
            new_physio = {}
            for modality, subject_list in sample["physio"].items():
                if modality not in self.scalers or not subject_list:
                    new_physio[modality] = subject_list
                else:
                    new_physio[modality] = [
                        self._transform_vec(v, modality) for v in subject_list
                    ]
            sample["physio"] = new_physio
        return dataset_list

from joblib import Parallel, delayed

def _transform_sample(sample, normalizer):
    new_physio = {}
    for modality, subject_list in sample["physio"].items():
        if modality not in normalizer.scalers or not subject_list:
            new_physio[modality] = subject_list
        else:
            new_physio[modality] = [normalizer._transform_vec(v, modality) for v in subject_list]
    return {**sample, "physio": new_physio}

def transform_dataset_parallel(dataset_list, normalizer, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=1)(
        delayed(_transform_sample)(s, normalizer) for s in dataset_list
    )
    return results

random.shuffle(dataset)

from sklearn.model_selection import train_test_split
from collections import Counter

def majority_vote(votes):
    counts = Counter(votes)
    return counts.most_common(1)[0][0]

labels_for_stratify = [majority_vote(x["label"]) for x in dataset]

train_dataset, val_dataset = train_test_split(
    dataset,
    test_size=0.15,
    stratify=labels_for_stratify,  # ajusta esto
    random_state=42
)
# train_size    = int(len(dataset) * 0.85)
# train_dataset = dataset[:train_size]
# val_dataset   = dataset[train_size:]

normalizer = GlobalPhysioNormalizer()
normalizer.fit(train_dataset)
train_dataset = transform_dataset_parallel(train_dataset, normalizer)
val_dataset   = transform_dataset_parallel(val_dataset,   normalizer)
# test_dataset = transform_dataset_parallel(test_dataset, normalizer)

os.makedirs("../data/last_task", exist_ok=True)

with open("../data/last_task/train.json", "w", encoding="utf-8") as f:
    json.dump(train_dataset, f, ensure_ascii=False, indent=2)
with open("../data/last_task/val.json", "w", encoding="utf-8") as f:
    json.dump(val_dataset, f, ensure_ascii=False, indent=2)
# with open("../data/processed/test.json", "w", encoding="utf-8") as f:
#     json.dump(test_dataset, f, ensure_ascii=False, indent=2)
 
# print(f"✅ test.json guardado → {len(test_dataset)} samples")

print(f"total: {len(dataset)} | train: {len(train_dataset)} | val: {len(val_dataset)}")


[GlobalPhysioNormalizer] ET: 12 columnas con log1p, 12 directas
[GlobalPhysioNormalizer] HR: 0 columnas con log1p, 4 directas
[GlobalPhysioNormalizer] EEG: 0 columnas con log1p, 80 directas


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 1740 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 2098 out of 2145 | elapsed:    1.3s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 2145 out of 2145 | elapsed:    1.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 332 out of 379 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 379 out of 379 | elapsed:    0.1s finished


total: 2524 | train: 2145 | val: 379
